# MorphIQ SDK for Python — Start Here

Price a **European call** and an **American put**, then try the async client.
These inputs are synthetic. No market-data subscription is needed.

This notebook makes real requests only when you execute the pricing cells.
Each sync cell makes one request; the async cell makes two more. Running all
cells therefore makes **four pricing requests**, which may consume your API
allowance. Opening Jupyter does not call the API.

Docker includes the required Python environment. For a downloaded notebook,
install this repository with its `notebooks` extra using the quickstart, then
open it in JupyterLab. The notebook is self-contained; it needs no examples imports.


In [ ]:
import sys

import morphiq_sdk
from morphiq_sdk import AsyncMorphIQ, MorphIQ

print("Python:", sys.version.split()[0])
print("MorphIQ SDK:", morphiq_sdk.__version__)

## Connect to your API

Get your API key from the
[MorphIQ Labs developer portal](https://developers.morphiqlabs.com).
The SDK connects to `https://api.morphiqlabs.com` automatically. Before starting
Jupyter or Docker Compose, set your API key in the terminal:

```sh
export MORPHIQ_API_KEY="YOUR_API_KEY"
```

Docker Compose passes the key into the notebook container. If it is unset,
the cell below prompts. The key prompt is masked: **do not paste your key into a code
cell**. The key stays in this kernel's process environment, not notebook source
or outputs; clear it in the cleanup cell and shut down the kernel when finished.

For a local or controlled test service, set `MORPHIQ_API_BASE_URL` before startup
to override the default API URL.

The following helpers are copied from the tested Python examples. Editing an
input later sends that value unchanged to the API; the server validates model
domains and selects the pricing algorithm.


In [ ]:
import json
import os
from getpass import getpass

from morphiq_sdk import PriceRequest, PriceResponse


def configure() -> None:
    """Read the API key from the environment, or prompt without displaying it."""
    if not os.environ.get("MORPHIQ_API_KEY"):
        os.environ["MORPHIQ_API_KEY"] = getpass("Your API key (hidden): ").strip()


def european_call() -> PriceRequest:
    """Synthetic one-year European call: 20% volatility and 5% interest."""
    return PriceRequest(
        exercise_style="european",
        option_type="call",
        spot=100.0,
        strike=100.0,
        time_to_expiry_years=1.0,
        risk_free_rate=0.05,
        dividend_yield=0.0,
        volatility=0.20,
    )


def american_put() -> PriceRequest:
    """Synthetic half-year American put with a 1% continuous dividend yield."""
    return PriceRequest(
        exercise_style="american",
        option_type="put",
        spot=100.0,
        strike=105.0,
        time_to_expiry_years=0.5,
        risk_free_rate=0.05,
        dividend_yield=0.01,
        volatility=0.20,
    )


def show_price(response: PriceResponse) -> None:
    """Display server results; an absent commercial charge is not zero."""
    print(
        json.dumps(
            {
                "price": response.result.price,
                "exercise_style": response.result.exercise_style,
                "option_type": response.result.option_type,
                "pricing_model": response.result.pricing_model,
                "model_version": response.result.model_version,
                "request_id": response.metadata.request_id,
                "engines": [engine.model_dump() for engine in response.metadata.engines],
                "commercial_cu_charged": (
                    response.metadata.fcu_charged
                    if response.metadata.fcu_charged is not None
                    else "not reported (for example, open local development)"
                ),
            },
            indent=2,
        )
    )

In [ ]:
configure()

## 1. European call

Spot and strike are both 100 in the same currency. Expiry is **1 year**,
volatility is **0.20 = 20%**, interest is **0.05 = 5%**, and dividend yield is zero.
The Python default sends `pricing_model="auto"`; read the returned model and
algorithm version. The price uses the same currency units as spot and strike;
no contract multiplier is applied.


In [ ]:
configure()
with MorphIQ() as client:
    response = client.price(european_call())
    show_price(response)

## 2. American put

Spot is 100, strike is 105, expiry is **0.5 years**, volatility is 20%, interest
is 5%, and the continuous dividend yield is **0.01 = 1%**. This is a separate
synthetic contract, not a like-for-like exercise-premium comparison with the call.
Cash-dividend schedules and date/day-count conversion are outside this first example.


In [ ]:
configure()
with MorphIQ() as client:
    response = client.price(american_put())
    show_price(response)

## 3. The same requests with async

Jupyter supports top-level `await`; do not use `asyncio.run` in a running notebook
event loop. These requests are deliberately sequential, allowing use with plans
that permit only one request at a time. This cell makes **two additional calls**.


In [ ]:
async def main() -> None:
    configure()
    async with AsyncMorphIQ() as client:
        # Sequential awaits fit plans that allow only one concurrent request.
        european = await client.price(european_call())
        show_price(european)
        american = await client.price(american_put())
        show_price(american)


await main()

## Reading results and failures

Use `response.result.price`, `response.result.pricing_model`, and
`response.metadata.request_id`. Commercial charges are `metadata.fcu_charged`;
if absent they are **unreported**, not zero. `units_charged` is a separate
internal compute weight. Keep the request ID when contacting support.

`APIError` exposes HTTP status, server code, message, request ID, optional charge
metadata and `Retry-After`. `RequestTimeout` means execution/charges may be
unknown. Both clients make **no automatic retries**, including after timeouts,
and do not follow redirects. Correct the input/key/endpoint or inspect the
reported charge before deliberately submitting another request.

## Finish

The next cell removes the API key from this kernel's environment. Shut down the
kernel to release its remaining memory. If you exported the key in your terminal,
run `unset MORPHIQ_API_KEY` there when finished; `docker compose down` removes the
container configured with that key. Clear outputs before sharing a notebook.


In [ ]:
if "MORPHIQ_API_KEY" in os.environ:
    del os.environ["MORPHIQ_API_KEY"]
print("API key removed from the kernel environment.")